← [Overview](00_overview.ipynb)

# Segmentation

Every step so far reduced the number of **periods**. Segmentation is the other axis: it reduces
the number of **timesteps inside** each period, by merging neighboring timesteps that look alike.
It is the last transform in the pipeline, and it never changes how many typical periods there are. It is also optional to apply segmentation at all.

It also acts on **one period at a time**, so — like [representation](03_representation.ipynb) —
a single period is the whole story.

| | |
|---|---|
| **In** | one typical period: `n_timesteps × n_attributes` |
| **Inside** | Ward merging, restricted to *adjacent* timesteps |
| **Out** | the same period in `n_segments` segments of **unequal length** |

> **Segmentation and `contiguous` clustering are the same algorithm.**
> Both are Ward agglomerative clustering with an adjacency constraint; they differ only in what
> they merge. [`contiguous`](02_clustering/02_agglomerative_clustering.ipynb) merges **periods**
> (rows of the D matrix); segmentation merges **timesteps** within a period. Both are
> *feature-based* — driven by value similarity, not by the clock.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import tsam
from tsam import ClusterConfig, SegmentConfig

pio.renderers.default = "notebook_connected"

ATTRS = ["solar", "load"]
N_TIMESTEPS = 4

tiny = pd.read_csv("../../data/tiny.csv", index_col=0, parse_dates=True)
D = pd.read_csv("../../data/tiny_periods.csv", header=[0, 1], index_col=0)

# Segmentation runs last, so its input is a finished set of typical periods.
base = tsam.aggregate(
    tiny,
    n_clusters=3,
    period_duration="1D",
    cluster=ClusterConfig(method="hierarchical"),
    preserve_column_means=False,
)
assignments = [int(c) for c in base.cluster_assignments]
print("k=3 Ward assignments:", assignments)
print("three typical periods to segment, one per cluster")

## 1  What comes in: one typical period

Segmentation sees a typical period the way clustering saw a day — but transposed. Clustering
treated a whole period as **one point in 8 dimensions**. Segmentation treats the same period as
**four points in 2 dimensions**: one point per timestep, its coordinates being the attributes.

Those four points are what get merged. We follow **typical period 1**, the sunny one.

In [ ]:
def medoid_index(matrix):
    """Index of the member closest to its cluster-mates (03's rule)."""
    dist = np.sqrt(((matrix[:, None, :] - matrix[None, :, :]) ** 2).sum(-1))
    return int(np.argmin(dist.sum(axis=0)))


TYPICAL = 1
members = [d for d, c in enumerate(assignments) if c == TYPICAL]
source_day = members[medoid_index(D.loc[members].values)]

# The typical period as timestep vectors: 4 rows (timesteps) x 2 columns (attributes).
period = D.loc[source_day].values.reshape(len(ATTRS), N_TIMESTEPS).T

print(f"typical period {TYPICAL} = medoid of days {members} = day{source_day}")
print("\nAs four points in attribute space (normalized):")
print(
    pd.DataFrame(
        period, columns=ATTRS, index=[f"t{t}" for t in range(N_TIMESTEPS)]
    ).round(4)
)
print("\nThe same period in physical units:")
print(base.cluster_representatives.loc[TYPICAL].to_string())

Read those four points as a day: dark at `t0`, full sun at `t1`, still bright at `t2`, dark again
at `t3` while load climbs. **`t1` and `t2` are nearly the same point** — that similarity is what
segmentation is about to exploit.

## 2  Inside: Ward merging, adjacent timesteps only

Start with every timestep in its own segment, then repeat: find the **adjacent** pair whose merge
costs least, and merge it. Stop at `n_segments`. The cost of merging segments $A$ and $B$ is the
Ward criterion — the within-segment variance the merge would create:

$$
\Delta(A, B) = \frac{|A| \cdot |B|}{|A| + |B|} \, \lVert \bar{x}_A - \bar{x}_B \rVert^2
$$

The adjacency restriction is the only thing separating this from ordinary clustering: `t0` may
merge with `t1`, never with `t3`. That is what keeps a segment a **contiguous block of time**
rather than a scattered set of hours.

In [ ]:
def ward_cost(a, b):
    """The within-segment variance created by merging adjacent segments a and b."""
    return (
        len(a)
        * len(b)
        / (len(a) + len(b))
        * ((a.mean(axis=0) - b.mean(axis=0)) ** 2).sum()
    )


segments = [[t] for t in range(N_TIMESTEPS)]
merge_history = []  # (left, right, cost) per accepted merge — the tree, drawn below
while len(segments) > 2:
    costs = [
        ward_cost(period[segments[i]], period[segments[i + 1]])
        for i in range(len(segments) - 1)
    ]
    print(
        f"{len(segments)} -> {len(segments) - 1} segments — cost of each adjacent merge:"
    )
    for i, cost in enumerate(costs):
        print(f"    t{segments[i]} + t{segments[i + 1]}: {cost:.4f}")
    cheapest = int(np.argmin(costs))
    print(f"    cheapest: merge t{segments[cheapest]} + t{segments[cheapest + 1]}\n")
    merge_history.append(
        (list(segments[cheapest]), list(segments[cheapest + 1]), costs[cheapest])
    )
    segments = [
        *segments[:cheapest],
        segments[cheapest] + segments[cheapest + 1],
        *segments[cheapest + 2 :],
    ]

print("segments:", segments)
print("durations:", tuple(len(s) for s in segments))

**The same tree, drawn.** That trace is a merge history, so it draws as a dendrogram — the same
view [contiguous clustering](02_clustering/02_agglomerative_clustering.ipynb) uses for its merge
history, which is what "the same algorithm" means in practice.

One difference from an ordinary dendrogram matters here: the leaves stay in **clock order**,
`t0 … t3`. A normal dendrogram is free to reorder its leaves to keep branches from crossing, but
under an adjacency constraint that would be misleading — only *neighbors* on this axis were ever
allowed to join, so every bracket below spans a contiguous block of time. Bar height is the Ward
cost of that merge.

In [ ]:
# Draw the merge history as a dendrogram, leaves pinned in clock order.
# Each node is (x, height): a leaf sits at its own timestep and height 0; a merged
# block sits at the midpoint of its children, at the cost that created it.
node = {(t,): (float(t), 0.0) for t in range(N_TIMESTEPS)}
top = max(cost for _, _, cost in merge_history)
cut = top * 1.3  # where the algorithm stopped, drawn above the last merge

fig = go.Figure()
for left, right, cost in merge_history:
    x_left, y_left = node[tuple(left)]
    x_right, y_right = node[tuple(right)]
    fig.add_trace(
        go.Scatter(
            x=[x_left, x_left, x_right, x_right],
            y=[y_left, cost, cost, y_right],
            mode="lines",
            line={"color": "#1f4ea1", "width": 2},
            hoverinfo="skip",
            showlegend=False,
        )
    )
    fig.add_annotation(
        x=(x_left + x_right) / 2,
        y=cost,
        text=f"<b>{cost:.4f}</b>",
        yshift=11,
        xshift=-34,  # clear of the riser that climbs from this bracket
        showarrow=False,
        font={"size": 11, "color": "#1f4ea1"},
    )
    node[tuple(left + right)] = ((x_left + x_right) / 2, cost)

# A segment that never merged has no bracket to draw it — carry its branch up.
for seg in segments:
    if len(seg) == 1:
        fig.add_trace(
            go.Scatter(
                x=[float(seg[0])] * 2,
                y=[0, cut],
                mode="lines",
                line={"color": "#1f4ea1", "width": 2, "dash": "dash"},
                hoverinfo="skip",
                showlegend=False,
            )
        )
    else:
        x_seg, y_seg = node[tuple(seg)]
        fig.add_trace(
            go.Scatter(
                x=[x_seg] * 2,
                y=[y_seg, cut],
                mode="lines",
                line={"color": "#1f4ea1", "width": 2},
                hoverinfo="skip",
                showlegend=False,
            )
        )

fig.add_hline(
    y=cut,
    line={"color": "#EF553B", "width": 2, "dash": "dot"},
    annotation_text="stop here — 2 segments",
    annotation_position="bottom left",
    annotation_font_color="#EF553B",
)
# What each branch crossing the cut became. Above the line, clear of the tree.
for seg in segments:
    fig.add_annotation(
        x=float(np.mean(seg)),
        y=cut,
        text=f"<b>segment t{seg}</b><br>duration {len(seg)}",
        yshift=26,
        showarrow=False,
        font={"size": 12, "color": "#00806b"},
    )

fig.update_layout(
    title=(
        "Segmentation dendrogram — only neighboring timesteps could merge<br>"
        "<sup>t1 and t2 are nearly the same point, so they join almost for free; "
        "t3 is the odd one out and never joins anything.</sup>"
    ),
    xaxis={
        "title": "timestep (clock order — only neighbors may merge)",
        "tickmode": "array",
        "tickvals": list(range(N_TIMESTEPS)),
        "ticktext": [f"t{t}" for t in range(N_TIMESTEPS)],
        "range": [-0.4, N_TIMESTEPS - 0.6],
    },
    yaxis={"title": "Ward cost of the merge", "range": [-0.02, cut * 1.35]},
    height=450,
)
fig.show()

The tree makes the argument the printout only implied. The near-identical `t1` and `t2` merge
first, at **0.0415** — a cost so far below the alternatives that their bracket sits almost on the
floor. On the second pass the new `[t1, t2]` block absorbs `t0` at **0.5138**, and `t3` — the one
timestep with high load — is never cheap enough to join anything before the algorithm stops.

## 3  What comes out: segments of unequal length

The result is **`[t0, t1, t2]` and `[t3]`: durations 3 and 1, not 2 and 2.**

This is the answer to the obvious question, and it is worth stating plainly: **segments are not
equal slices of the period.** Nothing in the algorithm balances them. A segment grows for exactly
one reason — the timesteps inside it resemble each other — so a segment is **long where the
profile is flat and short where it moves**. Segmentation is a *feature-based* reduction, not a
uniform downsample; `df.resample("6h").mean()` would have given 2 and 2 and thrown away the
distinction this preserves.

In [ ]:
segmented = tsam.aggregate(
    tiny,
    n_clusters=3,
    period_duration="1D",
    cluster=ClusterConfig(method="hierarchical"),
    segments=SegmentConfig(n_segments=2),
    preserve_column_means=False,
)

print(
    "hand-traced durations for typical period",
    TYPICAL,
    ":",
    tuple(len(s) for s in segments),
)
print(
    "tsam's durations for typical period",
    TYPICAL,
    ":",
    segmented.segment_durations[TYPICAL],
)
print("\nThe segmented typical periods — note the Segment Duration index:")
segmented.cluster_representatives.round(3)

### Unequal in a second way, too

The lengths differ **within** a period — and they also differ **between** periods. Segmentation
runs independently on each typical period, so each one is cut where *its own* profile happens to
be flat. There is no shared grid.

In [ ]:
print("segment durations, per typical period:")
for tp, durations in enumerate(segmented.segment_durations):
    print(f"  typical period {tp}: {durations}  (sums to {sum(durations)} timesteps)")

print("\nsame split for every period?", len(set(segmented.segment_durations)) == 1)

Typical period 0 happens to split evenly at 2 + 2; the other two split 3 + 1. Same `n_segments`,
different cuts — because they are different shapes.

Two consequences follow, and both matter downstream:

1. **A segment is not a timestep, and its value is not weighted like one.** Every segment carries
   a duration, exposed as `result.segment_durations` and as the `Segment Duration` level of the
   representative index. A model that sums over segments without weighting by duration will
   count the flat overnight block the same as the single peak hour — and get the wrong answer.
2. **Segments do not line up across typical periods.** There is no common time grid to index
   against, which is why the duration travels *with* the data rather than being implied by
   position.

With segmentation the pipeline is complete: periods grouped, condensed into representatives,
extremes protected, totals restored, and now the periods themselves compressed in time.

---

**Up next — putting it to work:**

* [Aggregate a time series](../../how-to/how_to_aggregate.ipynb) — the practical entry point
* [Optimization workflow](../../how-to/optimization_workflow.ipynb) — handing typical periods,
  counts and segment durations to a model

**See also:**

* [Contiguous clustering](02_clustering/02_agglomerative_clustering.ipynb) — the same algorithm,
  merging periods instead of timesteps
* [Representation](03_representation.ipynb) — `SegmentConfig(representation=…)` takes the same six
  rules, applied to the timesteps of a segment instead of the members of a cluster